In [ ]:
#celda para clasificar rápido algunos audios
from pathlib import Path
import numpy as np
import torch

NATURAL = [
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1138215.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1271820.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1272637.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1276960.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1341447.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1363611.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1596451.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1608170.flac",
]

GENERADO = [
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1004644.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1056709.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1195221.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1265032.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1287124.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1365409.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1477244.flac",
    r"C:\Users\gerbq\Downloads\LA\LA\ASVspoof2019_LA_train\flac\LA_T_1531370.flac",
]

@torch.no_grad()
def p1_for_file(fp: Path, win_sec=4.0, hop_sec=2.0) -> float:
    x = load_mono_16k(fp, 16000)
    ws = make_windows(x, 16000, win_sec, hop_sec)
    ps = []
    for w in ws:
        t = torch.from_numpy(w).unsqueeze(0).to(device)
        out = model(t)
        logits = out[1]  
        p1 = torch.softmax(logits, dim=1)[0, 1].item()
        ps.append(p1)
    return float(np.mean(ps))

def eval_group(name, paths):
    vals = []
    for p in paths:
        fp = Path(p)
        v = p1_for_file(fp)
        vals.append(v)
        print(f"{name:8s}  p1={v:0.6f}  file={fp.name}")
    vals = np.array(vals, dtype=float)
    return vals

print("---- NATURAL ----")
nat = eval_group("NATURAL", NATURAL)
print("\n---- GENERADO ----")
gen = eval_group("GENERADO", GENERADO)

print("\nResumen:")
print("NAT mean:", nat.mean(), "median:", np.median(nat), "p1>0.5:", (nat>0.5).sum(), "/", len(nat))
print("GEN mean:", gen.mean(), "median:", np.median(gen), "p1>0.5:", (gen>0.5).sum(), "/", len(gen))


---- NATURAL ----
NATURAL   p1=0.999995  file=LA_T_1138215.flac
NATURAL   p1=0.999990  file=LA_T_1271820.flac
NATURAL   p1=0.999999  file=LA_T_1272637.flac
NATURAL   p1=0.999996  file=LA_T_1276960.flac
NATURAL   p1=0.999999  file=LA_T_1341447.flac
NATURAL   p1=0.999998  file=LA_T_1363611.flac
NATURAL   p1=0.999998  file=LA_T_1596451.flac
NATURAL   p1=0.999994  file=LA_T_1608170.flac

---- GENERADO ----
GENERADO  p1=0.014480  file=LA_T_1004644.flac
GENERADO  p1=0.000156  file=LA_T_1056709.flac
GENERADO  p1=0.619288  file=LA_T_1195221.flac
GENERADO  p1=0.000013  file=LA_T_1265032.flac
GENERADO  p1=0.000103  file=LA_T_1287124.flac
GENERADO  p1=0.000273  file=LA_T_1365409.flac
GENERADO  p1=0.000124  file=LA_T_1477244.flac
GENERADO  p1=0.000260  file=LA_T_1531370.flac

Resumen:
NAT mean: 0.9999964237213135 median: 0.9999971389770508 p1>0.5: 8 / 8
GEN mean: 0.07933699962092078 median: 0.0002078496981994249 p1>0.5: 1 / 8


Consultando el main.py del repo, una probabilidad cerca de 1 sería bonafide y cerca de 0 spoof.

# 1. IMPORTS

In [2]:
from __future__ import annotations
import re, ast, sys, csv, math
from pathlib import Path
from typing import Any, Dict, List, Tuple
import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly
import librosa
import torch.nn.functional as F

# 2. RUTAS

In [3]:
REPO_DIR  = Path(r"C:\TFG\aasist") #ruta al repo AASIST
CONF_PATH = REPO_DIR / "config" / "AASIST.conf" #ruta al archivo de configuración
CKPT_PATH = REPO_DIR / "models" / "weights" / "AASIST.pth" #ruta al checkpoint del modelo  

INPUT_DIR = Path(r"C:\TFG\Datasets\Propio") #ruta a los audios
OUT_CSV   = Path(r"C:\TFG\Código\aasist_results.csv") #ruta al csv de salida

print("CONF exists:", CONF_PATH.exists(), CONF_PATH)
print("CKPT exists:", CKPT_PATH.exists(), CKPT_PATH)
print("INPUT exists:", INPUT_DIR.exists(), INPUT_DIR)

CONF exists: True C:\TFG\aasist\config\AASIST.conf
CKPT exists: True C:\TFG\aasist\models\weights\AASIST.pth
INPUT exists: True C:\TFG\Datasets\Propio


# 3. LEER CONF
El archivo .conf contiene arquitectura del modelo, parámetros de entrenamiento, dataset y su protocolo, preprocesado y checkpoints

In [4]:
txt = CONF_PATH.read_text(encoding="utf-8", errors="ignore")

def extract_enclosing_dict(text: str, anchor: str) -> str:
    i = text.find(anchor)
    if i == -1:
        raise RuntimeError(f"No encuentro el anchor {anchor!r} en el conf.")
    j = i
    while j >= 0 and text[j] != "{":
        j -= 1
    if j < 0:
        raise RuntimeError("No encuentro '{' antes del anchor.")

    depth = 0
    in_str = False
    esc = False
    k = j
    while k < len(text):
        ch = text[k]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[j:k+1]
        k += 1
    raise RuntimeError("No pude cerrar el bloque '{...}'.")

# 4. CONSTUIR d_args
d_args(arquitectura) contiene:
- first_conv: parámetros del primer bloque convolucional del modelo
- filts: estructura que marca cuántos canales/filtros usa el modelo en cada etapa/bloque
- get_dims: dimensiones internas del bloque del grafo de atención
- pool_ratios: lista de ratios usados para pooling
- temperatures: controla lo suave o agresiva que es una selección

In [5]:
block = extract_enclosing_dict(txt, '"first_conv"')
block = re.sub(r",\s*(\}|\])", r"\1", block)  # quita comas colgantes
d_args: Dict[str, Any] = ast.literal_eval(block)

required = ["filts", "gat_dims", "pool_ratios", "temperatures", "first_conv"]
missing = [k for k in required if k not in d_args]
print("d_args OK. Missing:", missing)
if missing:
    raise RuntimeError("No se extrajo el dict correcto del conf.")

d_args OK. Missing: []


# 5. INSTANCIAR EL MODELO

In [6]:
if str(REPO_DIR) not in sys.path: 
    sys.path.insert(0, str(REPO_DIR)) #añade la raiz del repo al path

from models.AASIST import Model

device = "cpu"
model = Model(d_args).to(device).eval()
print("Modelo instanciado")

Modelo instanciado


# 6. CARGAR CHEKPOINT
El checkpoint sirve para cargar los pesos del entrenamiento

In [7]:
state = torch.load(str(CKPT_PATH), map_location=device)
if isinstance(state, dict):
    for key in ("state_dict", "model", "model_state_dict", "net"):
        if key in state and isinstance(state[key], dict):
            state = state[key]
            break

if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)
print(f"Checkpoint cargado (missing={len(missing_keys)}, unexpected={len(unexpected_keys)})")

Checkpoint cargado (missing=0, unexpected=0)


# 7. HELPERS DE AUDIO + VENTANA DESLIZANTE
Los helpers sirven para dejar los audios en un modelo estándar: mono(un canal), 16 Khz.
La ventana sirve para audios más largos de 4 segundos, los parte en ventanas de 4 segundos con solapamiento de dos para que no se quede información en el borde de una ventana.

In [1]:
import numpy as np
import soundfile as sf
import librosa

def load_mono_16k(path, target_sr=16000):
    x, sr = sf.read(str(path), always_2d=False)
    x = x.astype(np.float32)
    if x.ndim > 1:
        x = np.mean(x, axis=1).astype(np.float32)
    if sr != target_sr:
        x = librosa.resample(x, orig_sr=sr, target_sr=target_sr).astype(np.float32)
    x = np.clip(x, -1.0, 1.0).astype(np.float32)

    return x


def make_windows(x: np.ndarray, sr=16000, win_sec=4.0, hop_sec=2.0):
    win = int(win_sec * sr)
    hop = int(hop_sec * sr)
    if len(x) <= win:
        y = np.zeros(win, dtype=np.float32)
        y[:len(x)] = x
        return [y]
    out=[]
    for s in range(0, len(x)-win+1, hop):
        out.append(x[s:s+win])
    if (len(x)-win) % hop != 0:
        out.append(x[-win:])
    return out

# 8. INFERENCIA

In [8]:
#Este hace media de probabilidades de todas las ventanas
@torch.no_grad()
def prob1_mean_for_file(wav: Path, win_sec=4.0, hop_sec=2.0):
    x = load_mono_16k(wav, 16000)
    ws = make_windows(x, 16000, win_sec, hop_sec)
    ps=[]
    for w in ws:
        t = torch.from_numpy(w).unsqueeze(0).to(device)  # (1,T)

        out = model(t)
        if isinstance(out, (tuple, list)):
            logits = next(o for o in out if torch.is_tensor(o) and o.ndim == 2 and o.shape[1] == 2)
        else:
            logits = out


        p1 = torch.softmax(logits, dim=1)[0,1].item()
        ps.append(p1)

    return float(np.mean(ps)), len(x)/16000.0, len(ws) #devuelve la probabilidad media contando todas las ventanas

In [9]:
#Este sería sin ventanas
from data_utils import pad 

CUT = 64600  # ~4.04s 

@torch.no_grad()
def score_for_file_repo_style(wav: Path):
    x = load_mono_16k(wav, 16000)              
    x4 = pad(x, CUT).astype(np.float32)      

    t = torch.from_numpy(x4).unsqueeze(0).to(device)  

    out = model(t)
    if isinstance(out, (tuple, list)):
        logits = next(o for o in out if torch.is_tensor(o) and o.ndim == 2 and o.shape[1] == 2)
    else:
        logits = out

    score_bonafide = logits[0, 1].item()

    prob_bonafide = F.softmax(logits, dim=1)[0, 1].item()

    duration_sec = len(x) / 16000.0
    return score_bonafide, prob_bonafide, duration_sec


In [10]:
#Este sumando los logists de todas las ventanas antes de hacer la media
from data_utils import pad

CUT = 64600

@torch.no_grad()
def multiwindow_prob_bonafide(wav: Path, win_sec=4.0, hop_sec=2.0):
    x = load_mono_16k(wav, 16000)
    ws = make_windows(x, 16000, win_sec, hop_sec)

    logits_sum = None

    for w in ws:
        w = pad(w, CUT).astype(np.float32)
        t = torch.from_numpy(w).unsqueeze(0).to(device)

        _, logits = model(t) 

        logits_sum = logits if logits_sum is None else logits_sum + logits

    prob_bona = F.softmax(logits_sum, dim=1)[0, 1].item()
    score_bona = logits_sum[0, 1].item() 

    return prob_bona, score_bona, len(x)/16000.0, len(ws)


# 9. TRATAR LOS AUDIOS

In [11]:
files = sorted(INPUT_DIR.rglob("*.wav"))
print("WAVs encontrados:", len(files))
if not files:
    raise RuntimeError("No hay .wav en INPUT_DIR (ni en subcarpetas).")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["path", "prob1_mean", "duration_sec", "n_windows"])
    for i, fp in enumerate(files, 1):
        try:
            #p1, dur, nw = prob1_mean_for_file(fp, win_sec=4.0, hop_sec=2.0)
            #prob_bona, score_bona, dur, nw = multiwindow_prob_bonafide(fp, win_sec=4.0, hop_sec=2.0)
            #w.writerow([str(fp), prob_bona, dur, nw])
            score_bona, prob_bona, dur = score_for_file_repo_style(fp)
            w.writerow([str(fp), prob_bona, dur])
        except Exception as e:
            print(" Error con", fp, "->", repr(e))
        if i % 25 == 0:
            print(f"{i}/{len(files)}...")

print(" Terminado. CSV en:", OUT_CSV)

WAVs encontrados: 200
25/200...
50/200...
75/200...
100/200...
125/200...
150/200...
175/200...
200/200...
 Terminado. CSV en: C:\TFG\Código\aasist_results.csv
